# Notebook to design a method to find the Circoscription with an Address

## Step 1 : Normalize and Geolocate the address

API of the data base Adresse Nationale (BAN)

API Géoplateforme - Géocodage :
- user guide https://www.data.gouv.fr/dataservices/api-geoplateforme-geocodage
- swagger : https://data.geopf.fr/geocodage/openapi

In [1]:
#name of the server containing the geocoding service
server_name = "https://data.geopf.fr/geocodage"

## Search service of the Géoplateforme - Géocodage API
with GET Method

A noter : limite de 50 requêtes par seconde depuis une même adresse IP

In [10]:
#get method with requests library
import requests

#address to geocode
address = "castets-en-dorthe"

response = requests.get(
    server_name + "/search", #endpoint of the geocoding service
    params={
        "q": address,
        "format": "json",
        "index": "address,poi", #index to search in, can be address or poi (point of interest or admin unit)
        "limit": "10" #max number of results to return
    }
)

response.raise_for_status() #raise an error if the request failed

test_address_geocode = response.json()

print(test_address_geocode)

{'type': 'FeatureCollection', 'features': [{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-0.149547, 44.556404]}, 'properties': {'name': ['Castets-en-Dorthe'], 'toponym': 'Castets-en-Dorthe', 'category': ['lieu-dit habité', "zone d'habitation"], 'postcode': ['33210'], 'citycode': ['33106'], 'depcode': ['33'], 'city': ['Castets et Castillon'], 'extrafields': {'cleabs': 'PAIHABIT0000000050824604'}, 'classification': 7, 'territory': 'METROPOLE', 'score': 0.8545454545454545, '_type': 'poi'}}, {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-0.146662, 44.559672]}, 'properties': {'label': 'Route Jean-Baptiste de Baudre 33210 Castets et Castillon', 'score': 0.6870354545454544, 'id': '33106_0039', 'banId': '1665bc10-dd8e-4611-83fd-c9c0ea8ef88e', 'name': 'Route Jean-Baptiste de Baudre', 'postcode': '33210', 'citycode': '33106', 'oldcitycode': '33106', 'x': 450162.13, 'y': 6389489.22, 'city': 'Castets et Castillon', 'oldcity': 'Castets-en-Dorthe', 'context': 

Print pretty the Json returned

In [11]:
import json

print(json.dumps(test_address_geocode, indent=4, ensure_ascii=False))

{
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [
                    -0.149547,
                    44.556404
                ]
            },
            "properties": {
                "name": [
                    "Castets-en-Dorthe"
                ],
                "toponym": "Castets-en-Dorthe",
                "category": [
                    "lieu-dit habité",
                    "zone d'habitation"
                ],
                "postcode": [
                    "33210"
                ],
                "citycode": [
                    "33106"
                ],
                "depcode": [
                    "33"
                ],
                "city": [
                    "Castets et Castillon"
                ],
                "extrafields": {
                    "cleabs": "PAIHABIT0000000050824604"
                },
       

Retrieve coordinates

In [4]:
#from the response converted in json, we extract the coordinates
test_address_geocode_coordinates = test_address_geocode["features"][0]["geometry"]["coordinates"]

print(test_address_geocode_coordinates)

[2.330587, 48.868425]


## Step 2: Get the circo from the coordinates

### Method 1 : download locally GEOJson map and find polygon containing the coordinates

Reading the shp file donwloaded from https://www.data.gouv.fr/datasets/portraits-des-circonscriptions-legislatives-indicateurs-economiques-et-socio-demographiques

In [12]:
import geopandas as gpd

gpf = gpd.read_file("countours_files/circonscriptions_legislatives_030522.shp")

print(gpf.head())

  id_circo  dep                           libelle  \
0    97302  973       Guyane - 2e circonscription   
1    97301  973      Guyane - 1re circonscription   
2    97201  972  Martinique - 1re circonscription   
3    97202  972   Martinique - 2e circonscription   
4    97203  972   Martinique - 3e circonscription   

                                            geometry  
0  MULTIPOLYGON (((-54.60236 2.33356, -54.60242 2...  
1  MULTIPOLYGON (((-51.93691 4.46648, -51.93667 4...  
2  MULTIPOLYGON (((-61.00495 14.57791, -61.00492 ...  
3  MULTIPOLYGON (((-61.12889 14.63162, -61.1289 1...  
4  POLYGON ((-61.03945 14.64265, -61.0394 14.6425...  


In [13]:
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon

point = Point(test_address_geocode_coordinates[0], test_address_geocode_coordinates[1])

for i in range(len(gpf)):
    polygon = gpf.loc[i, 'geometry']
    if polygon.contains(point):
        print(f"The point is inside the polygon with index {i}. The circonscription is {gpf.loc[i, 'libelle']} et son uid est {gpf.loc[i, 'id_circo']}.")
        break

The point is inside the polygon with index 75. The circonscription is Paris - 1re circonscription et son uid est 7501.


### Method 2 : Use Service API to return circo with GPS coordinates

Services found so far : 
- Managed by SciencePO : https://services-eu1.arcgis.com/bFTOZwh6Wo1fgzl7/ArcGIS/rest/services/Circonscriptions_legislatives_France/FeatureServer
- Managed by SciencePO : https://services-eu1.arcgis.com/bFTOZwh6Wo1fgzl7/ArcGIS/rest/services/Circonscriptions_legislatives_FranceT1_T2/FeatureServer

For those 2 services, the method to identify the Circo seems to be the same as in method 1

To Inspect the service feature :  use the following query https://services-eu1.arcgis.com/bFTOZwh6Wo1fgzl7/ArcGIS/rest/services/Circonscriptions_legislatives_France/FeatureServer//0?f=pjson

To download the map : append `/0/query` and specify parameters such as `where`, `outFields`, `returnGeometry`, and `f=json`
 https://services-eu1.arcgis.com/bFTOZwh6Wo1fgzl7/ArcGIS/rest/services/Circonscriptions_legislatives_France/FeatureServer/0/query?where=1%3D1&outFields=*&returnGeometry=true&f=json

## Step 3 : How to deal with multiple results sent by the Geocodage API

In [14]:
#address to geocode
address_multi = "castets-en-dorthe, 33120, France"

response_multi = requests.get(
    server_name + "/search", #endpoint of the geocoding service
    params={
        "q": address_multi,
        "format": "json",
        "index": "address,poi", #index to search in, can be address or poi (point of interest or admin unit)
        "limit": "10" #max number of results to return
    }
)

test_address_multi_geocode = response_multi.json()

#print pretty json
print(json.dumps(test_address_multi_geocode, indent=4, ensure_ascii=False))


{
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [
                    -0.152642,
                    44.562103
                ]
            },
            "properties": {
                "label": "Rue du Ra 33210 Castets et Castillon",
                "score": 0.47026363636363633,
                "id": "33106_0070",
                "banId": "30008b96-d8cd-4f23-81e7-484e7491fc42",
                "name": "Rue du Ra",
                "postcode": "33210",
                "citycode": "33106",
                "oldcitycode": "33106",
                "x": 449698.36,
                "y": 6389777.99,
                "city": "Castets et Castillon",
                "oldcity": "Castets-en-Dorthe",
                "context": "33, Gironde, Nouvelle-Aquitaine",
                "type": "street",
                "importance": 0.4479,
                "depcode": "33",
 

In [15]:
# create dictionnary with labels and coordinates of the results
results_dict = {}

for i in range(len(test_address_multi_geocode["features"])):

    name = test_address_multi_geocode["features"][i]["properties"]["name"]
    if type(name) is list :
        name = name[0] #get the first name of the result
    coords = test_address_multi_geocode["features"][i]["geometry"]["coordinates"]
    
    results_dict[name] = coords

print("Results:", results_dict)


Results: {'Rue du Ra': [-0.152642, 44.562103], 'Route Jean Douence': [-0.151516, 44.556648], 'Rue Benoit': [-0.154102, 44.55775], 'Impasse du Brana': [-0.154131, 44.554844], 'Route du Grava': [-0.132538, 44.553882], 'Impasse la Grange': [-0.140395, 44.55592], 'Place du Casse': [-0.153743, 44.557156], 'Rue du Port': [-0.1556, 44.563142], 'Route de Langon': [-0.156773, 44.561326], 'Impasse Cantau': [-0.121436, 44.549544]}


In [16]:
# find the circonscription for each result
for label, coords in results_dict.items():
    point = Point(coords[0], coords[1])
    for i in range(len(gpf)):
        polygon = gpf.loc[i, 'geometry']
        if polygon.contains(point):
            print(f"The point '{label}' is inside the polygon with index {i}. The circonscription is {gpf.loc[i, 'libelle']} et son uid est {gpf.loc[i, 'id_circo']}.")
            break

The point 'Rue du Ra' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Route Jean Douence' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Rue Benoit' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Impasse du Brana' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Route du Grava' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Impasse la Grange' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Place du Casse' is inside the polygon with index 141. The circonscription is Gironde - 9e circonscription et son uid est 3309.
The point 'Rue du Port' is inside 